In [0]:
%sql
-- Create the target table

-- SCD Type 2 target table
-- This table will maintain historical records

CREATE OR REPLACE TABLE customer_target (
    customer_id INT,
    name STRING,
    city STRING,
    email STRING,

    -- When this version of the record became active
    start_date DATE,

    -- When this version became inactive
    end_date DATE,

    -- TRUE = latest/current record
    -- FALSE = historical record
    is_current BOOLEAN
)
USING DELTA;

In [0]:
%sql
-- Insert initial customer data
INSERT INTO customer_target
VALUES
(101, 'John',  'Mumbai', 'john@gmail.com',
 '2026-01-01', NULL, TRUE),

(102, 'Alice', 'Pune', 'alice@gmail.com',
 '2026-01-01', NULL, TRUE),

(103, 'David', 'Delhi', 'david@gmail.com',
 '2026-01-01', NULL, TRUE);

In [0]:
%sql
SELECT *
FROM customer_target
ORDER BY customer_id;

In [0]:
%sql
-- Create the source table
CREATE OR REPLACE TABLE customer_source (
    customer_id INT,
    name STRING,
    city STRING,
    email STRING
)
USING DELTA;

In [0]:
%sql
-- Insert new source data
INSERT INTO customer_source
VALUES
(101, 'John',   'Pune',  'john@gmail.com'),
(102, 'Alice',  'Pune',  'alice@gmail.com'),
(103, 'David',  'Delhi', 'david@gmail.com'),
(104, 'Robert', 'Mumbai', 'robert@gmail.com');

In [0]:
%sql
SELECT *
FROM customer_source
ORDER BY customer_id;

In [0]:
%sql
-- Find changed and new records
CREATE OR REPLACE TEMP VIEW customer_changes AS

SELECT
    s.customer_id,
    s.name,
    s.city,
    s.email,

    CASE
        -- Customer does not exist in target
        WHEN t.customer_id IS NULL
            THEN 'NEW'

        -- Customer exists but some data changed
        WHEN NOT (
            s.name <=> t.name
            AND s.city <=> t.city
            AND s.email <=> t.email
        )
            THEN 'CHANGED'

        -- Customer exists and nothing changed
        ELSE 'NO_CHANGE'
    END AS change_type

FROM customer_source s

LEFT JOIN customer_target t
    ON s.customer_id = t.customer_id
    AND t.is_current = TRUE;

In [0]:
%sql
SELECT *
FROM customer_changes
ORDER BY customer_id;

In [0]:
%sql
-- Expire the old record
MERGE INTO customer_target AS target

USING customer_changes AS source

ON target.customer_id = source.customer_id
AND target.is_current = TRUE

WHEN MATCHED
AND source.change_type = 'CHANGED'

THEN UPDATE SET
    target.end_date = current_date(),
    target.is_current = FALSE;

In [0]:
%sql
-- Insert the new version
INSERT INTO customer_target
SELECT
    customer_id,
    name,
    city,
    email,

    -- New version starts today
    current_date() AS start_date,

    -- Current record has no end date
    NULL AS end_date,

    -- Mark as current
    TRUE AS is_current

FROM customer_changes

WHERE change_type IN ('CHANGED', 'NEW');

In [0]:
%sql
-- Check final result
SELECT *
FROM customer_target
ORDER BY customer_id, start_date;